In [27]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import scipy.ndimage
from scipy.spatial.distance import cdist
from tqdm import tqdm  # 引入tqdm来显示进度条

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ==============================================================================
# --- 1. 配置 (请确认这里的路径) ---
# ==============================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"
# 已划分的数据集根目录 (包含JPEGImages, Annotations, ImageSets)
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/BUSI_for_SAM2"
# 测试集列表文件路径
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets","val.txt")


# -----------------baseline---------------------
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2/checkpoints/sam2.1_hiera_base_plus.pt"
model = build_sam2(SAM2_CFG_NAME, SAM2_CHECKPOINT_PATH, device=device)
predictor = SAM2ImagePredictor(model)

# -----------------adapter----------------------
# SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/busi_adapter_run2/checkpoints/checkpoint.pt"



# # bLoss
# # SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/busi_bLossv2_adapter_run5/checkpoints/checkpoint.pt"
# HYDRA_OVERRIDES = [
#     "model.image_encoder.trunk._target_=sam2.modeling.backbones.hieradet_adapterv1.Hiera",
#     "+model.image_encoder.trunk.use_adapter=True",
#     "+model.use_adapter=True",
# ]
# print("Loading SAM2 model in Adapter mode...")
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = build_sam2(
#     SAM2_CFG_NAME, 
#     SAM2_CHECKPOINT_PATH, 
#     device=device,
#     hydra_overrides_extra=HYDRA_OVERRIDES
# )
# predictor = SAM2ImagePredictor(model)
# print(f"Model and Predictor loaded on {device}.")


# ==============================================================================
# --- 3. 定义指标计算函数 (新增HD95) ---
# ==============================================================================
def calculate_hd95(gt_mask, pred_mask):
    """手动实现95% Hausdorff Distance (HD95)"""
    gt_bool = gt_mask > 0
    pred_bool = pred_mask > 0
    
    # 边缘情况：如果预测为空或者GT为空，返回NaN
    if np.sum(gt_bool) == 0 or np.sum(pred_bool) == 0:
        return np.nan
        
    # 提取边界
    gt_borders = np.logical_xor(gt_bool, scipy.ndimage.binary_dilation(gt_bool))
    pred_borders = np.logical_xor(pred_bool, scipy.ndimage.binary_dilation(pred_bool))
    
    # 获取边界像素点的坐标
    gt_coords = np.column_stack(np.where(gt_borders))
    pred_coords = np.column_stack(np.where(pred_borders))
    
    if len(gt_coords) == 0 or len(pred_coords) == 0:
        return np.nan
        
    # 计算预测边界点到GT边界点的所有成对距离
    dists = cdist(pred_coords, gt_coords, metric='euclidean')
    
    # 预测点到GT的最短距离，以及GT点到预测的最短距离
    min_dist_pred_to_gt = np.min(dists, axis=1)
    min_dist_gt_to_pred = np.min(dists, axis=0)
    
    # 取两者的95%分位数，并返回其中的最大值
    hd95 = max(np.percentile(min_dist_pred_to_gt, 95), np.percentile(min_dist_gt_to_pred, 95))
    return hd95

def calculate_metrics(gt_mask, pred_mask):
    """计算Dice系数、IoU和HD95"""
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0
    
    # Dice 和 IoU
    intersection = np.logical_and(gt_mask_bool, pred_mask_bool).sum()
    dice = (2. * intersection) / (gt_mask_bool.sum() + pred_mask_bool.sum() + 1e-8)
    union = gt_mask_bool.sum() + pred_mask_bool.sum() - intersection
    iou = intersection / (union + 1e-8)
    
    # HD95
    hd95 = calculate_hd95(gt_mask, pred_mask)
    
    return dice, iou, hd95

# ==============================================================================
# --- 4. 遍历测试集、执行分割并评估 ---
# ==============================================================================
if not os.path.exists(TEST_SET_FILE):
    raise FileNotFoundError(f"测试集定义文件未找到: {TEST_SET_FILE}")

with open(TEST_SET_FILE, 'r') as f:
    test_sample_names = [line.strip() for line in f.readlines()]

print(f"\nFound {len(test_sample_names)} samples in the test set. Starting evaluation...")

results = []
images_dir = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
annotations_dir = os.path.join(SPLIT_DATASET_ROOT, "Annotations")

for sample_name in tqdm(test_sample_names, desc="Evaluating Test Set"):
    image_path = os.path.join(images_dir, sample_name, "00000.jpg")
    mask_path = os.path.join(annotations_dir, sample_name, "00000.png")
    
    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"Warning: Files for sample '{sample_name}' not found. Skipping.")
        continue

    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    predictor.set_image(image_rgb)
    contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print(f"Warning: No lesion found in mask for {sample_name}. Skipping.")
        continue
        
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    box_prompt = np.array([[x, y, x + w, y + h]])

    masks, scores, logits = predictor.predict(
        box=box_prompt,
        multimask_output=False
    )
    
    pred_mask = (masks[0] * 255).astype(np.uint8)
    
    # 获取指标
    dice, iou, hd95 = calculate_metrics(gt_mask, pred_mask)
    
    category = 'benign' if 'benign' in sample_name else 'malignant'
    results.append({
        "category": category,
        "sample_name": sample_name,
        "dice": dice,
        "iou": iou,
        "hd95": hd95
    })

    predictor.reset_predictor()

# ==============================================================================
# --- 5. 打印最终的总结报告 ---
# ==============================================================================
if results:
    df = pd.DataFrame(results)
    
    # 1. 按类别计算平均值和标准差
    category_summary = df.groupby('category').agg(
        mean_dice=('dice', 'mean'),
        std_dice=('dice', 'std'),
        mean_iou=('iou', 'mean'),
        std_iou=('iou', 'std'),
        mean_hd95=('hd95', 'mean'),
        std_hd95=('hd95', 'std'),
        count=('sample_name', 'count')
    ).reset_index()
    
    # 2. 计算所有样本的整体平均值 (Overall)
    overall_summary = pd.DataFrame([{
        'category': 'overall',
        'mean_dice': df['dice'].mean(),
        'std_dice': df['dice'].std(),
        'mean_iou': df['iou'].mean(),
        'std_iou': df['iou'].std(),
        'mean_hd95': df['hd95'].mean(),  # pandas的.mean()会自动忽略预测全空的NaN情况
        'std_hd95': df['hd95'].std(),
        'count': len(df)
    }])
    
    # 将 overall 拼接到原来的 summary 中
    final_summary = pd.concat([category_summary, overall_summary], ignore_index=True)
    
    print("\n" + "="*80)
    print("--- Evaluation Summary on Test Set (val.txt) ---")
    print("="*80)
    # 使用 round(4) 让输出表格更美观
    print(final_summary.round(4).to_string(index=False))
    print("="*80)

else:
    print("No images were processed. Please check paths and file structure.")

print("\n--- Evaluation on the test set is complete! ---")


Found 130 samples in the test set. Starting evaluation...


Evaluating Test Set: 100%|██████████| 130/130 [00:14<00:00,  8.94it/s]


--- Evaluation Summary on Test Set (val.txt) ---
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9103    0.0820    0.8429   0.1023    14.1516   13.3976     91
malignant     0.8736    0.0532    0.7793   0.0812    39.4455   26.7143     39
  overall     0.8993    0.0762    0.8238   0.1005    21.7398   21.6990    130

--- Evaluation on the test set is complete! ---


lambdas = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0, 10.0, 20.0]
dice = [0.9221, 0.9248, 0.9342, 0.9347, 0.9346, 0.9335, 0.9334, 0.9315, 0.9260, 0.9091]
hd95 = [14.5528, 14.2134, 13.2045, 12.9993, 12.6389, 14.0464, 13.8381, 14.1183, 14.4969, 16.0519]
baseline_dice = 0.8996
baseline_hd95 = 21.7398

category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  
baseline -no adapter -no Bloss
overall     0.8996    0.0762    0.8238   0.1005    21.7398       
adapter -no bloss
overall     0.9301    0.0339    0.8711   0.0572    14.5528     
λ 0.5
overall     0.9248    0.0495    0.8607   0.0761    14.2134   
λ 1.0
overall     0.9342    0.0295    0.8779   0.0508    13.2045   
λ 1.5
overall     0.9347    0.0297    0.8788   0.0510    12.9993   
λ 2.0
overall     0.9346    0.0295    0.8788   0.0507    12.6389    
λ 2.5
overall     0.9335    0.0316    0.8770   0.0538    14.0464
λ 3.0
overall     0.9334    0.0311    0.8767   0.0531    13.8381   
λ 5.0
overall     0.9315    0.0318    0.8734   0.0544    14.1183   
λ 10.0
overall     0.9260    0.0344    0.8640   0.0579    14.4969   
λ 20.0
overall     0.9091    0.0403    0.8358   0.0661    16.0519


#run10 0.2  20
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9141    0.0377    0.8440   0.0625     9.3164    5.7677     91
malignant     0.8974    0.0440    0.8166   0.0710    31.7678   18.5892     39
  overall     0.9091    0.0403    0.8358   0.0661    16.0519   15.2209    130

#run14 0.1  10
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9324    0.0314    0.8749   0.0536     8.3671    7.0646     91
malignant     0.9111    0.0367    0.8387   0.0602    28.7999   16.9814     39
  overall     0.9260    0.0344    0.8640   0.0579    14.4969   14.4264    130

#run15 0.05 5
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9393    0.0279    0.8867   0.0484     8.2080    7.7375     91
malignant     0.9134    0.0333    0.8423   0.0554    27.9089   15.3275     39
  overall     0.9315    0.0318    0.8734   0.0544    14.1183   13.8965    130

#run16 0.01 1
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9432    0.0243    0.8934   0.0427     8.0790    7.4967     91
malignant     0.9133    0.0303    0.8418   0.0504    28.4972   17.9378     39
  overall     0.9342    0.0295    0.8779   0.0508    14.2045   14.9071    130

#run17 0.03 3
   benign     0.9420    0.0253    0.8915   0.0443     7.8804    6.9709     91
malignant     0.9133    0.0343    0.8421   0.0562    27.7392   17.1816     39
  overall     0.9334    0.0311    0.8767   0.0531    13.8381   14.2941    130

#run18 0.02 2
   benign     0.9433    0.0243    0.8937   0.0429     7.9071    7.1185     91
malignant     0.9146    0.0309    0.8440   0.0510    26.8464   15.3677     39
  overall     0.9347    0.0295    0.8788   0.0507    13.5889   13.4474    130

#run19 0.015 1.5
   benign     0.9434    0.0242    0.8939   0.0426     7.8004    6.8482     91
malignant     0.9142    0.0315    0.8434   0.0522    28.4634   18.1701     39
  overall     0.9347    0.0297    0.8788   0.0510    13.9993   14.8435    130

#0.5

#v2_run1 0.15 1.5
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9419    0.0253    0.8912   0.0443     8.1511    7.6264     91
malignant     0.9151    0.0319    0.8449   0.0525    27.6793   17.5515     39
  overall     0.9338    0.0300    0.8773   0.0513    14.0096   14.5612    130

#v2_run2 1.5
 category  mean_dice  std_dice  mean_iou  std_iou  mean_hd95  std_hd95  count
   benign     0.9406    0.0276    0.8891   0.0477     8.8066    9.0642     91
malignant     0.9081    0.0327    0.8332   0.0536    29.4808   16.6290     39
  overall     0.9308    0.0327    0.8723   0.0556    15.0089   15.1404    130

#v2_run5 2.5
   benign     0.9421    0.0255    0.8916   0.0447     7.6473    6.2107     91
malignant     0.9135    0.0354    0.8427   0.0583    28.9777   17.2793     39
  overall     0.9335    0.0316    0.8770   0.0538    14.0464   14.5310    130

